In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl

sys.path.append('../src')

from data_pipeline import load_all_raw_data

from data_analysis import (
    filter_data_until_date, temporal_split_data, 
)

from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)

from preprocess import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)

from weather_data import WeatherDataCollector


⚠️ Warning: Testing module not available. Pipeline will run without validation.


In [3]:
# Load the same dataset used in softmax-ETA.ipynb
df_feat = pl.read_parquet("../data/processed/ecobici_features.parquet")
print("Dataset shape:", df_feat.shape)

Dataset shape: (12976053, 130)


In [4]:
# ============================================
# LightGBM ETA regression model (GPU ready)
# ============================================
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# -------- 1. Conversión Polars → pandas -----------------
print("\nPreparing dataset for LightGBM...")

df = df_feat.to_pandas()           # df_feat proviene de tu pipeline previo
df["fecha_origen_recorrido"] = pd.to_datetime(df["fecha_origen_recorrido"])

# -------- 2. Definición de columnas ---------------------
cat_cols = [
    "id_estacion_origen", "modelo_bicicleta", "genero",
    "weather_weather_code_x", "weather_is_day_x",
    "is_holiday_ar", "is_weekend", "payday_flag",
    "vacation_season", "peak_commute", "precip_flag",
    "hour", "dow", "month", "day"
]

cont_cols = [
    "long_estacion_origen", "lat_estacion_origen",
    "weather_temperature_2m_x", "weather_relative_humidity_2m_x",
    "weather_dew_point_2m_x", "weather_apparent_temperature_x",
    "weather_precipitation_x", "weather_rain_x",
    "weather_pressure_msl_x", "weather_surface_pressure_x",
    "weather_cloud_cover_x", "weather_cloud_cover_low_x",
    "weather_cloud_cover_mid_x", "weather_cloud_cover_high_x",
    "weather_et0_fao_evapotranspiration_x",
    "weather_vapour_pressure_deficit_x",
    "weather_wind_speed_10m_x", "weather_wind_gusts_10m_x",
    "weather_soil_temperature_0_to_7cm_x",
    "weather_soil_moisture_0_to_7cm_x",
    "weather_sunshine_duration_x", "weather_direct_radiation_x",
    "wind_dir_sin", "wind_dir_cos", "precipitation_lag_1h", "rain_lag_1h",
    "weather_code_cat_Clear", "weather_code_cat_Clouds",
    "weather_code_cat_Drizzle", "weather_code_cat_Rain",
    "dep_last_DT", "trip_dur_mean_last_DT", "model_FIT_cnt",
    "model_ICONIC_cnt", "share_male", "share_female", "share_other",
    "dep_lag_1", "dep_lag_2", "dep_lag_3", "dep_lag_4", "dep_lag_5",
    "dep_lag_6", "arr_last_DT", "arr_lag_1", "arr_lag_2", "arr_lag_3",
    "arr_lag_4", "arr_lag_5", "arr_lag_6", "dep_ma_24h", "dep_std_24h",
    "dep_ratio_DT_24h", "near_dep_sum_DT", "near_dep_lag_1",
    "sin_hour", "cos_hour", "sin_dow", "cos_dow", "sin_month", "cos_month"
]

# Mantener solo columnas presentes
cat_cols  = [c for c in cat_cols  if c in df.columns]
cont_cols = [c for c in cont_cols if c in df.columns]
feature_cols = cat_cols + cont_cols
target_col = "duracion_recorrido"

# -------- 3. Encoding categórico ------------------------
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# -------- 4. Train / Val split temporal -----------------
split_date = pd.Timestamp("2023-12-01")
train_df = df[df["fecha_origen_recorrido"] < split_date]
val_df   = df[df["fecha_origen_recorrido"] >= split_date]
print(f"Train size: {train_df.shape}, Val size: {val_df.shape}")

# -------- 5. Datasets LightGBM --------------------------
lgb_train = lgb.Dataset(
    train_df[feature_cols],
    label=train_df[target_col],
    categorical_feature=cat_cols
)
lgb_val = lgb.Dataset(
    val_df[feature_cols],
    label=val_df[target_col],
    categorical_feature=cat_cols
)

# -------- 6. Hiper-parámetros ---------------------------
params = {
    "objective": "regression",
    "metric": ["rmse", "mae"],
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

# Detectar automáticamente qué clave usar para GPU según versión
if hasattr(lgb, "__version__"):
    major = int(lgb.__version__.split(".")[0])
    gpu_key = "device_type" if major >= 4 else "device"
else:
    gpu_key = "device"

params[gpu_key] = "gpu"
params["max_bin"] = 255            #  ≤ 255 obligatorio en GPU

# -------- 7. Callbacks (early stopping & logging) -------
callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100)
]



Preparing dataset for LightGBM...
Train size: (10609550, 130), Val size: (2366503, 130)

Training LightGBM model...


LightGBMError: bin size 374 cannot run on GPU

In [6]:
params["max_bin"] = 255            #  ≤ 255 obligatorio en GPU

In [8]:
params.pop(gpu_key, None)      # quita la clave GPU
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    callbacks=callbacks
)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[16]	train's rmse: 14388.2	train's l1: 688.173	valid's rmse: 10624.1	valid's l1: 785.958


In [7]:

# -------- 8. Entrenamiento ------------------------------
print("\nTraining LightGBM model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    callbacks=callbacks
)

# -------- 9. Métricas de validación ---------------------
val_pred = model.predict(val_df[feature_cols])
print("\nValidation metrics:")
print(f"RMSE: {mean_squared_error(val_df[target_col], val_pred, squared=False):.4f}")
print(f"MAE:  {mean_absolute_error(val_df[target_col], val_pred):.4f}")
print(f"R²:   {r2_score(val_df[target_col], val_pred):.4f}")

# -------- 10. Importancia de características ------------
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importance()
}).sort_values("importance", ascending=False)

print("\nTop 10 most important features:")
print(importance.head(10))

# -------- 11. Guardar modelo ----------------------------
model.save_model("lgbm_eta_model.txt")
print("\n✔ Model saved as lgbm_eta_model.txt")


Training LightGBM model...


LightGBMError: bin size 374 cannot run on GPU

In [ ]:
# ============================================
# LightGBM Classification for Destination Prediction
# ============================================
print("\n" + "="*50)
print("DESTINATION CLASSIFICATION TASK")
print("="*50)

# Use the same features but predict destination station
target_cls = "id_estacion_destino"

# Label-encode the destination target
le_dest = LabelEncoder()
df[target_cls] = le_dest.fit_transform(df[target_cls].astype(str))

print(f"Number of unique destination stations: {len(le_dest.classes_)}")

# Same temporal split
train_df_cls = df[df["fecha_origen_recorrido"] < split_date]
val_df_cls = df[df["fecha_origen_recorrido"] >= split_date]

# Prepare LightGBM datasets for classification
lgb_train_cls = lgb.Dataset(train_df_cls[feature_cols], label=train_df_cls[target_cls],
                           categorical_feature=cat_cols)
lgb_val_cls = lgb.Dataset(val_df_cls[feature_cols], label=val_df_cls[target_cls],
                         categorical_feature=cat_cols)

# Classification parameters
params_cls = {
    "objective": "multiclass",
    "num_class": len(le_dest.classes_),
    "metric": ["multi_logloss"],
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

# Try to add GPU support
try:
    params_cls["device_type"] = "gpu"
    _ = lgb.train(params_cls, lgb_train_cls, num_boost_round=10)
    print("✓ GPU found – training classification on GPU")
except Exception as e:
    params_cls["device_type"] = "cpu"
    print("✗ GPU not available – falling back to CPU for classification")

# Train classification model
print("\nTraining LightGBM classification model...")

# Setup callbacks for classification
callbacks_cls = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100)
]

model_cls = lgb.train(
    params_cls,
    lgb_train_cls,
    num_boost_round=1000,
    valid_sets=[lgb_train_cls, lgb_val_cls],
    valid_names=["train", "valid"],
    callbacks=callbacks_cls
)

# Classification metrics
from sklearn.metrics import accuracy_score, classification_report

val_pred_cls = model_cls.predict(val_df_cls[feature_cols])
val_pred_cls_labels = np.argmax(val_pred_cls, axis=1)

print("\nClassification metrics:")
print(f"Accuracy: {accuracy_score(val_df_cls[target_cls], val_pred_cls_labels):.4f}")

# Top-5 accuracy (since there are many stations)
def top_k_accuracy(y_true, y_pred_proba, k=5):
    top_k_pred = np.argsort(y_pred_proba, axis=1)[:, -k:]
    correct = sum([y_true[i] in top_k_pred[i] for i in range(len(y_true))])
    return correct / len(y_true)

print(f"Top-5 Accuracy: {top_k_accuracy(val_df_cls[target_cls].values, val_pred_cls, k=5):.4f}")
print(f"Top-10 Accuracy: {top_k_accuracy(val_df_cls[target_cls].values, val_pred_cls, k=10):.4f}")

# Feature importance for classification
importance_cls = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_cls.feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features for destination prediction:")
print(importance_cls.head(10))

# Save classification model
model_cls.save_model("lgbm_destination_classifier.txt")
print("\n✔ Classification model saved as lgbm_destination_classifier.txt")

# Save label encoder for destination
import joblib
joblib.dump(le_dest, "destination_label_encoder.pkl")
joblib.dump(encoders, "feature_encoders.pkl")
print("✔ Label encoders saved")


In [ ]:
# ============================================
# Multi-Task Learning Summary & Comparison
# ============================================
print("\n" + "="*50)
print("MULTI-TASK LEARNING SUMMARY")
print("="*50)

print("\n🎯 REGRESSION TASK (ETA Prediction):")
print(f"   RMSE: {mean_squared_error(val_df[target_col], val_pred, squared=False):.4f}")
print(f"   MAE:  {mean_absolute_error(val_df[target_col], val_pred):.4f}")
print(f"   R²:   {r2_score(val_df[target_col], val_pred):.4f}")

print(f"\n🎯 CLASSIFICATION TASK (Destination Prediction):")
print(f"   Accuracy:      {accuracy_score(val_df_cls[target_cls], val_pred_cls_labels):.4f}")
print(f"   Top-5 Acc:     {top_k_accuracy(val_df_cls[target_cls].values, val_pred_cls, k=5):.4f}")
print(f"   Top-10 Acc:    {top_k_accuracy(val_df_cls[target_cls].values, val_pred_cls, k=10):.4f}")
print(f"   Num Classes:   {len(le_dest.classes_)}")

print("\n📊 FEATURE IMPORTANCE COMPARISON:")
print("\nTop 5 features for ETA prediction:")
for i, row in importance.head(5).iterrows():
    print(f"   {row['feature']}: {row['importance']:.0f}")

print("\nTop 5 features for destination prediction:")
for i, row in importance_cls.head(5).iterrows():
    print(f"   {row['feature']}: {row['importance']:.0f}")

print("\n💾 SAVED MODELS:")
print("   ✓ lgbm_eta_model.txt (regression)")
print("   ✓ lgbm_destination_classifier.txt (classification)")
print("   ✓ destination_label_encoder.pkl")
print("   ✓ feature_encoders.pkl")

print(f"\n📈 DATASET INFO:")
print(f"   Total samples: {len(df):,}")
print(f"   Train samples: {len(train_df):,}")
print(f"   Val samples: {len(val_df):,}")
print(f"   Features: {len(feature_cols)}")
print(f"   Categorical: {len(cat_cols)}")
print(f"   Continuous: {len(cont_cols)}")

print("\n🚀 Models ready for deployment!")
